[Guide Index](README.md)


# 04b SHACL and SPARQL

Two ways SHACL 1.2 SPARQL Extensions let a shape reach directly into SPARQL: `sh:sparql`, an ad hoc constraint whose violations are computed by a query, and a user-defined `sh:ConstraintComponent`, a reusable, named constraint type backed by a SPARQL ASK or SELECT validator. Both are defined by SHACL 1.2 SPARQL Extensions; `starlayer.shacl` evaluates the queries with RDF 1.2 awareness (triple-term syntax, `isTRIPLE()`, etc. work inside either one) and supports `sh:resultAnnotation`.

The data and shapes graphs in the examples are `StarLayerGraph` objects (the working name of what becomes `StarLayerGraph`; see the [Oxigraph wrapper guide](02d-oxigraph-wrapper.ipynb)). `starlayer.shacl` accepts them, runs the queries below itself (not through `StarLayerGraph.query()`), and returns the report as an `StarLayerGraph` (pyoxigraph terms).

See the [SHACL shapes guide](04-shacl-shapes.ipynb) for how this fits alongside `starlayer.shacl`'s other features, and the [SPARQL guide](03-sparql.ipynb) for the query functions available inside these queries.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections reuse variables from earlier ones.


In [1]:
from rdflib import Literal, Namespace
from starlayer.graph.oxgraph import StarLayerGraph
from starlayer.shacl import StarShaclSchema

# EX, SH and Literal are rdflib terms: they are accepted as input by StarLayerGraph.add(), and as pattern input when
# looking terms up in the report, which is an StarLayerGraph (see the SHACL shapes guide, section 1.1).
EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

## 1. `sh:sparql` constraints

`sh:sparql` attaches an ad hoc SPARQL-based constraint directly to a shape. The `sh:select` query pre-binds `$this` (the focus node); each row the query returns is one violation. No parameter, no reusable component — for that, see section 2.

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:score 80 .
    ex:bob ex:score 20 .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ScoreShape a sh:NodeShape ;
      sh:targetSubjectsOf ex:score ;
      sh:sparql [
        sh:message "Score must be at least 50" ;
        sh:select "PREFIX ex: <http://example.org/> SELECT $this WHERE { $this ex:score ?s . FILTER(?s < 50) }" ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False)
print("conforms:", result.conforms)
print("ex:bob flagged:", "Focus Node: ex:bob" in result.report_text)
print("ex:alice flagged:", "Focus Node: ex:alice" in result.report_text)

conforms: False
ex:bob flagged: True
ex:alice flagged: False


### 1.1 RDF 1.2 syntax inside the query

The query text is run with StarLayer's SPARQL 1.2 handling for the graph `starlayer.shacl` validates — triple-term patterns (`<<( )>>`), `isTRIPLE()`, and the rest of the RDF-1.2-aware function set (see the [SPARQL guide](03-sparql.ipynb)) all work inside `sh:select`.

In [3]:
data = StarLayerGraph()
data.add((EX.alice, EX.says, (EX.bob, EX.knows, EX.carol)))

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix ex: <http://example.org/> .
    ex:SaysTripleTermShape a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:sparql [
        a sh:SPARQLConstraint ;
        sh:message "alice must say a triple term about bob knowing carol" ;
        sh:select \"\"\"
          PREFIX ex: <http://example.org/>
          SELECT $this WHERE {
            FILTER NOT EXISTS {
              $this ex:says <<( ex:bob ex:knows ex:carol )>> .
            }
          }
        \"\"\" ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False)
print("conforms (alice says the right triple term):", result.conforms)

data_wrong = StarLayerGraph()
data_wrong.add((EX.alice, EX.says, (EX.bob, EX.likes, EX.dana)))
result_wrong = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data_wrong, meta_shacl=False)
print("conforms (alice says a different triple term):", result_wrong.conforms)

conforms (alice says the right triple term): True
conforms (alice says a different triple term): False


### 1.2 `sh:resultAnnotation`: inject extra properties into a violation

A SHACL 1.2 SPARQL Extensions feature (`sh:resultAnnotation`). `sh:annotationVarName` sources a value from the SELECT query's own result row (here, the actual score that failed); `sh:annotationValue` (not shown) would attach the same fixed value to every result instead.

In [4]:
data = StarLayerGraph()
data.add((EX.alice, EX.score, Literal(20)))
data.add((EX.bob, EX.score, Literal(80)))

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ; sh:targetSubjectsOf ex:score ;
      sh:sparql [
        sh:select "PREFIX ex: <http://example.org/> SELECT $this ?actualScore WHERE { $this ex:score ?actualScore . FILTER(?actualScore < 50) }" ;
        sh:resultAnnotation [
          sh:annotationProperty ex:reportedScore ;
          sh:annotationVarName "actualScore" ;
        ] ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False)
print("conforms:", result.conforms)
for _, _, value in result.report_graph.triples((None, EX.reportedScore, None)):
    print("annotated reportedScore:", int(value.value))

conforms: False
annotated reportedScore: 20


## 2. User-defined `sh:ConstraintComponent`

A reusable, named constraint type: declare `a sh:ConstraintComponent` with `sh:parameter` (the extra predicate a shape uses to configure it) and `sh:validator`/`sh:nodeValidator`/`sh:propertyValidator` (a SPARQL ASK or SELECT query deciding conformance). Once declared, the parameter (`ex:minScore` below) is just an ordinary predicate any shape can use.

In [5]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:score 80 .
    ex:bob ex:score 20 .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:MinScoreConstraintComponent a sh:ConstraintComponent ;
      sh:parameter [ sh:path ex:minScore ] ;
      sh:validator [
        a sh:SPARQLAskValidator ;
        sh:ask "ASK { FILTER (?value >= ?minScore) }" ;
      ] .
    ex:ScoreShape a sh:NodeShape ;
      sh:targetSubjectsOf ex:score ;
      sh:property [ sh:path ex:score ; ex:minScore 50 ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False)
print("conforms:", result.conforms)
print("ex:bob flagged:", "Focus Node: ex:bob" in result.report_text)
print("ex:alice flagged:", "Focus Node: ex:alice" in result.report_text)

conforms: False
ex:bob flagged: True
ex:alice flagged: False


`sh:validator` (used above, node-scoped) has two property-scoped counterparts: `sh:propertyValidator` (an ASK/SELECT over each value node individually) and `sh:nodeValidator` (over the focus node itself, ignoring the property path) — same declaration shape, different binding scope. And a SELECT-based validator works the same way as `sh:sparql`'s own SELECT form: each returned row is one violation, with `$this`/`?value` pre-bound.

In [6]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:MinScoreConstraintComponent a sh:ConstraintComponent ;
      sh:parameter [ sh:path ex:minScore ] ;
      sh:propertyValidator [
        a sh:SPARQLSelectValidator ;
        sh:select "SELECT $this ?value WHERE { FILTER (?value < ?minScore) }" ;
      ] .
    ex:ScoreShape a sh:NodeShape ;
      sh:targetSubjectsOf ex:score ;
      sh:property [ sh:path ex:score ; ex:minScore 50 ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False)
print("conforms:", result.conforms)
print(result.report_text)

conforms: True
Validation Report
Conforms: True



### 2.1 RDF 1.2 values inside a custom validator

The same RDF-1.2-aware query rewriting applies here too — `isTRIPLE()` correctly distinguishes a real triple-term value from a plain IRI.

In [7]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:claims <<( ex:bob ex:age 42 )>> .
    ex:carol ex:claims ex:not_a_triple .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:MustBeTripleTermComponent a sh:ConstraintComponent ;
      sh:parameter [ sh:path ex:mustBeTripleTerm ] ;
      sh:validator [
        a sh:SPARQLAskValidator ;
        sh:ask "ASK { FILTER (isTRIPLE(?value)) }" ;
      ] .
    ex:ClaimShape a sh:NodeShape ;
      sh:targetSubjectsOf ex:claims ;
      sh:property [ sh:path ex:claims ; ex:mustBeTripleTerm true ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False)
print("conforms:", result.conforms)
print("ex:carol flagged (plain IRI, not a triple term):", "Focus Node: ex:carol" in result.report_text)
print("ex:alice flagged:", "Focus Node: ex:alice" in result.report_text)

conforms: False
ex:carol flagged (plain IRI, not a triple term): True
ex:alice flagged: False


## Known limitations

- **Calling a custom node expression function as an ordinary SPARQL function by name** from `sh:select`/`sh:sparqlExpr` query text (e.g. `ex:instanceCount(ex:Name)` inside a `BIND(...)`) — a separate mechanism from the node-expression call forms in the [node expressions guide](04a-shacl-node-expressions.ipynb#7.-Extending-the-vocabulary:-custom-node-expression-functions) — is not implemented; registering a user-defined function as a real SPARQL engine function is a larger mechanism than what's built today.
- **Content-level validation of `sh:select`/`sh:ask`/`sh:construct` query text** (is the string actually syntactically valid SPARQL, not just an `xsd:string`) is checked structurally by meta-shacl today, not parsed as real SPARQL — a deliberate scope decision, see the [SHACL shapes guide](04-shacl-shapes.ipynb#Known-limitations).